# 03 – Time Series Demand Forecasting

This notebook will explore time series models and workflows for demand forecasting using the `src/forecasting` and `src/data` packages.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.synthetic_generators import generate_demand_series
from src.forecasting.time_series_models import train_arima, forecast_horizon
from src.optimization.resource_allocation import optimize_allocation_from_forecast

%matplotlib inline

In [ ]:
# Generate synthetic daily demand and split into history and forecast horizon
n_periods = 200
horizon = 30

values = generate_demand_series(n_periods, random_state=0)
index = pd.date_range(start="2024-01-01", periods=n_periods, freq="D")
demand_series = pd.Series(values, index=index, name="demand")

history = demand_series.iloc[:-horizon]
actual_future = demand_series.iloc[-horizon:]

In [ ]:
# Plot historical demand
ax = history.plot(figsize=(10, 4), title="Synthetic historical demand")
ax.set_ylabel("Units")
plt.show()

In [ ]:
# Train ARIMA on the historical demand and forecast the next horizon
model = train_arima(history, order=(2, 1, 2))
forecast = forecast_horizon(model, steps=horizon)

# Plot history, actual future, and forecast
ax = history.plot(label="history", figsize=(10, 4))
actual_future.plot(ax=ax, label="actual future")
forecast.plot(ax=ax, label="forecast", linestyle="--")
ax.set_title("Demand forecast with ARIMA (history vs actual vs forecast)")
ax.set_ylabel("Units")
ax.legend()
plt.show()

In [ ]:
# Use the forecast as input to a simple resource allocation LP
# Here we plan for the average daily demand over the forecast horizon.
capacities = np.array([150.0, 120.0, 80.0])

result = optimize_allocation_from_forecast(
    forecast=forecast,
    capacities=capacities,
    unit_cost=1.0,
    aggregation="mean",
)

In [ ]:
# Inspect the optimization result
print("Status:", result.status)
print("Objective value:", result.objective_value)
print("Allocation matrix (resources x products):")
print(result.allocation)
print("Resource slack (remaining capacity):", result.resource_slack)
print("Product slack (oversupply):", result.product_slack)

In [ ]:
# Visualize allocation versus capacity for each resource
resources = np.arange(len(capacities))
allocated = result.allocation[:, 0]
capacity_values = capacities

width = 0.35
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(resources - width / 2, capacity_values, width, label="capacity")
ax.bar(resources + width / 2, allocated, width, label="allocated")
ax.set_xticks(resources)
ax.set_xticklabels([f"R{i+1}" for i in resources])
ax.set_ylabel("Units")
ax.set_title("Resource capacity vs allocated production")
ax.legend()
plt.show()